# Final Binary Baseline Summary

This notebook consolidates the final binary classification stage of the diabetic retinopathy project.
It combines the strongest binary baselines, the clinical error comparison, the final baseline selection, and the explainability summary for the selected model.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

In [ ]:
project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / "src").exists() and (candidate / "configs").exists():
        project_root = candidate
        break

comparison_dir = project_root / "outputs" / "comparison_all"
final_dir = project_root / "outputs" / "final_binary_summary"
final_dir.mkdir(parents=True, exist_ok=True)

gradcam_dir = project_root / "outputs" / "efficientnet_b0_baseline" / "gradcam"
b4_dir = project_root / "outputs" / "efficientnet_b4_baseline"

print(f"Project root: {project_root}")
print(f"Final summary directory: {final_dir}")

**Table 17.1. Final binary metric comparison across the strongest baseline candidates**

In [ ]:
metrics_df = pd.read_csv(comparison_dir / "table_6_1_all_model_metrics.csv")
b4_metrics_df = pd.read_csv(b4_dir / "table_16_1_efficientnet_b4_metrics.csv")

b4_metric_map = dict(zip(b4_metrics_df["metric"], b4_metrics_df["value"]))
metrics_df["EfficientNet-B4"] = metrics_df["metric"].map(b4_metric_map)

model_order = [
    "Custom CNN",
    "ResNet18",
    "ResNet18 + Preprocessing",
    "DenseNet121",
    "EfficientNet-B0",
    "EfficientNet-B4",
]

final_metric_df = metrics_df[["metric", *model_order]].copy()
final_metric_df = final_metric_df.round(6)
final_metric_df.to_csv(final_dir / "table_17_1_final_binary_metrics_comparison.csv", index=False)
final_metric_df

**Table 17.2. Clinical error comparison for the strongest binary baselines**

In [ ]:
clinical_df = pd.read_csv(comparison_dir / "table_6_3_clinical_summary.csv")
b4_confusion_df = pd.read_csv(b4_dir / "table_16_2_efficientnet_b4_confusion_counts.csv")
b4_confusion_map = dict(zip(b4_confusion_df["outcome"], b4_confusion_df["count"]))

b4_row = pd.DataFrame([
    {
        "model": "EfficientNet-B4",
        "recall": b4_metric_map["recall"],
        "precision": b4_metric_map["precision"],
        "f1_score": b4_metric_map["f1_score"],
        "false_negative": int(b4_confusion_map["False Negative"]),
        "false_positive": int(b4_confusion_map["False Positive"]),
    }
])

final_clinical_df = pd.concat([clinical_df, b4_row], ignore_index=True)
final_clinical_df = final_clinical_df.round(6)
final_clinical_df.to_csv(final_dir / "table_17_2_final_binary_clinical_summary.csv", index=False)
final_clinical_df

**Table 17.3. Final binary model selection summary**

In [ ]:
selection_df = pd.DataFrame([
    {
        "criterion": "Best overall binary baseline",
        "selected_model": "EfficientNet-B0",
        "reason": "Best accuracy, precision, F1-score, and near-best ROC-AUC with only 1 false positive.",
    },
    {
        "criterion": "Best recall-oriented screening model",
        "selected_model": "ResNet18",
        "reason": "Highest recall and lowest false-negative count among the binary baselines.",
    },
    {
        "criterion": "Largest EfficientNet tested",
        "selected_model": "EfficientNet-B4",
        "reason": "Did not outperform EfficientNet-B0 under the current setup despite the larger backbone.",
    },
])
selection_df.to_csv(final_dir / "table_17_3_final_binary_selection_summary.csv", index=False)
selection_df

**Figure 17.1. Metric comparison across the final binary baseline candidates**

In [ ]:
plot_metric_order = ["accuracy", "precision", "recall", "f1_score", "roc_auc"]
plot_df = final_metric_df[final_metric_df["metric"].isin(plot_metric_order)].copy()
plot_df["metric"] = pd.Categorical(plot_df["metric"], categories=plot_metric_order, ordered=True)
plot_df = plot_df.sort_values("metric")
plot_df = plot_df.melt(id_vars="metric", var_name="model", value_name="value")

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=plot_df, x="metric", y="value", hue="model", ax=ax)
ax.set_title("Final Binary Baseline Metric Comparison")
ax.set_xlabel("Metric")
ax.set_ylabel("Value")
ax.set_ylim(0.85, 1.01)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(final_dir / "figure_17_1_final_binary_metric_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

**Figure 17.2. False-positive and false-negative comparison across the main binary baselines**

In [ ]:
error_plot_df = final_clinical_df[["model", "false_negative", "false_positive"]].melt(
    id_vars="model",
    var_name="error_type",
    value_name="count",
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=error_plot_df, x="model", y="count", hue="error_type", ax=ax)
ax.set_title("Clinical Error Comparison Across Binary Baselines")
ax.set_xlabel("Model")
ax.set_ylabel("Count")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(final_dir / "figure_17_2_final_binary_error_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

**Table 17.4. Explainability outcome summary for the selected final binary baseline `EfficientNet-B0`**

In [ ]:
gradcam_summary_df = pd.read_csv(gradcam_dir / "table_10_1_efficientnet_b0_outcome_summary.csv")
gradcam_summary_df.to_csv(final_dir / "table_17_4_efficientnet_b0_explainability_summary.csv", index=False)
gradcam_summary_df

**Image 17.1. Consolidated Grad-CAM panel for the selected final binary baseline `EfficientNet-B0`**

In [ ]:
image_specs = [
    ("True Positive", gradcam_dir / "image_10_1_efficientnet_b0_true_positive_gradcam.png"),
    ("True Negative", gradcam_dir / "image_10_2_efficientnet_b0_true_negative_gradcam.png"),
    ("False Positive", gradcam_dir / "image_10_3_efficientnet_b0_false_positive_gradcam.png"),
    ("False Negative", gradcam_dir / "image_10_4_efficientnet_b0_false_negative_gradcam.png"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (title, image_path) in zip(axes.flat, image_specs):
    image = Image.open(image_path)
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
fig.savefig(final_dir / "image_17_1_efficientnet_b0_gradcam_panel.png", dpi=300, bbox_inches="tight")
plt.show()

**Table 17.5. Final binary interpretation summary for the thesis**

In [ ]:
interpretation_df = pd.DataFrame([
    {
        "topic": "Final selected model",
        "summary": "EfficientNet-B0 is the strongest overall binary baseline in the current project.",
    },
    {
        "topic": "Clinical tradeoff",
        "summary": "ResNet18 remains the most recall-oriented model, but EfficientNet-B0 offers a stronger overall balance with fewer false positives.",
    },
    {
        "topic": "Explainability insight",
        "summary": "Grad-CAM for EfficientNet-B0 is generally retinal-centered and more clinically plausible than the scratch model, but some artifact sensitivity remains.",
    },
    {
        "topic": "Architecture scaling insight",
        "summary": "EfficientNet-B4 did not outperform EfficientNet-B0 in this setup, showing that a larger backbone is not automatically better.",
    },
])
interpretation_df.to_csv(final_dir / "table_17_5_final_binary_interpretation_summary.csv", index=False)
interpretation_df